# Ingestion of Crop Data

## Environment Setup and Configuration

### Test kernel

Just for checking the environment is working.

In [2]:
txt = "Hello"
print(txt)

Hello


### Import Required Libraries

In [3]:
# For calling data repository URL
import requests
import io

# For manipulating data with Pandas
import pandas as pd

# Ensure the odfpy library is installed
%pip install odfpy

Note: you may need to restart the kernel to use updated packages.


## Objective

(no code) Sourcing and formatting crop data for subsequent analytical exploration.

### Thoughts

- A clean dataframe of the specific data that I want.
- Maybe export to new cleansed CSV file ready for ingestion into (SQLite) database.
- Think the crop data may only be avilable as csv downloads.
- Might think to automate download, but in the first instance, just download and process.
- Crop yield data is available for several crops, over many years, by each region.
- Paired data is for weather at a regional level.

### Plan

(no code) Steps identified for sourcing and formatting crop data.

### General Plan

1. Get data from url.
2. Parse into dataframe for manipulation.
  2.1 The dataframe could be output to csv.
3. Format data to useful dataframe for analysis.
4. Conduct exploratory data analysis to identify interesting features for exploration.
5. Analyse interesting feature interactions with statisitical models/ML.
6. Explore interesing interactions with further EDA and ML.

- Direct retrival of data from url
  - Perhaps download raw then process?
  - Alternatively direct manipulation for smaller dataset.
- Ingestion into dataframe.
- Cleansing through manipulation of dataframe.
- Output cleansed CSV for next process in the pipeline.

- ETL
  - Extract
  - Transform
  - Load

- ELT
  - Extract
  - Load
  - Tranform

### Region Labels

Need to consider how the data is going to be aggregated. This needs to match with the aggregation of weather data, so will have to aggregate to the lowest common denominator.

Initial EDA may hihglight regional or country level variation highlighting features for exploration.

#### Crop Data Grouping Lists

These lists show how the data is presented in the source data.

<table>

<tr><th>Whole List</th><th>UK Countries</th><th>UK Only Regions</th></tr>
<tr><td>

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

</td><td>

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

</td><td>

|ID|Crop Regions|
|--|------------|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|

</td></tr> </table>

## Extract Data from Government Online Repository 

- Source data from URL.
- Parse into Pandas DataFrame.
- List sheets in ODF workbook.
- For each sheet in ODF workbook:
  - Extract as dataframe.
  - Export to local storage as CSV.

### Source data from Government repository

Try following url: <https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods>

Objects:

- label: **buffer_object**, object type: **_io.BytesIO**

In [ ]:
# N.B. Need to ensure odfpy is installed in the environment. This allows the ODS binary to be parsed.

# Creates a ByteIO Class Object

import requests
import io

url = "https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods"
response = requests.get(url)
response.raise_for_status() # This gives the REST request response.

buffer_object = io.BytesIO(response.content)

In [ ]:
type(buffer_object)

### Identify the sheets in the workbook

Objects:
- label: **list_odf_sheets**, object type: **list**
- label: **sheets_list**, object typeL **.csv**


In [ ]:
# https://pandas.pydata.org/docs/dev/reference/api/pandas.ExcelFile.sheet_names.html
# this outputs an ExcelFile Class Object

import pandas as pd

crops_file = pd.ExcelFile(
				path_or_buffer=buffer_object
				,engine="odf")

list_odf_sheets: list = []

for sheet in crops_file.sheet_names:
	list_odf_sheets.append(sheet)

list_odf_sheets

Output list to a .csv using Pandas

In [ ]:
df_odf_list = pd.DataFrame(list_odf_sheets)

df_odf_list.to_csv('data/crops/sheet_list.csv', index=False)

df_odf_list.head()

### Parse each sheet into separate .csv

For each sheet with suffix 'Region' parse into from **buffer_object** into dataframes and output as local .csv files.

Objects:
- label: **raw_wheat.csv**, object type: **.csv**
- label: **raw_winter_barley.csv**, object type: **.csv**
- label: **raw_spring_barley.csv**, object type: **.csv**
- label: **raw_total_barley.csv**, object type: **.csv**
- label: **raw_oats.csv**, object type: **.csv**
- label: **raw_osr.csv**, object type: **.csv**

Locally saving the file allows decoupling the need for network calls, allowing for faster I/O operations. This also allows me to continue development whilst internet connection was not availalble, e.g. whilst travelling on the train.

Exporting locally to .csv files does come at the cost of storage. This can be mitigated by deleting the files once processing is completed, retaining only the required data for reference. However, the size of data is very small and easily maintained on even the most limited of systems.

In [ ]:
# Define the prefix
prefix = 'Region'

# Use for loop to list through list of sheet names, parse sheet into dataframe, and output locally as .csv file.
for name in list_odf_sheets:
	if name.startswith(prefix):
		df_sheet = (pd.ExcelFile(path_or_buffer=buffer_object,engine="odf")).parse(name)
		df_sheet.to_csv(f'data/crops/{name}.csv', index=False)

## Clean data

- For each sheet CSV:
  - load into dataframe.
  - crop to new dataframe with relevant rows and columns.
  - Reset column headers and index to years.

### Load single crop .csv into DataFrame

In [ ]:
# Will start with the Regional_spring_barley.csv
df_spring_barley = pd.read_csv('data/crops/Regional_spring_barley.csv')

df_spring_barley

### Read specific worksheet into Dataframe

Get data from specific worksheet with the data

In [ ]:
df_spring_barley.iloc[10: , :3]

In [ ]:
df_spring_barley.iloc[:5, 23:]

In [ ]:
df_spring_barley.iloc[10: ,23: ]

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html
df_crops = pd.read_excel(buffer_object, engine="odf", header=None)

print(f'Dataframe head data is:\n{df_crops.head()}')

#### With read_excel

In [ ]:

df_wheat = pd.read_excel(
    buffer_object,
    sheet_name="Regional_wheat",
    engine="odf", header=None
)

print(df_wheat.head())

#### With ExcelFile

In [ ]:
print(df_wheat.columns)

#### Get just the rows that show Wheat Yield Data

In [ ]:
df_wheat_yields = df_wheat.loc[17:30]
print(df_wheat_yields.head())

In [ ]:
print(df_wheat_yields.columns)

#### Get data rows for spring barley

In [ ]:
df_spring_barley_yields = df_spring_barley.loc[16:30]
print(df_spring_barley_yields)

In [ ]:
df_spring_barley_yields.columns

In [ ]:
df_spring_barley_yields.iloc[0]

In [ ]:
df_spring_barley_yields.columns = df_spring_barley_yields.iloc[0]

### Reset column header row

Reset the header row to be the new row 0, rather than have this considered a data row.

In [ ]:
# https://saturncloud.io/blog/convert-the-first-row-of-a-pandas-dataframe-to-column-names-a-comprehensive-guide/

df_wheat_yields.columns = df_wheat_yields.iloc[0] 	# specifies row 0, first row, as being the column names.
df_yields = df_wheat_yields[1:]						# resests the dataframe data as starting from the second row, index 1 onwards ':'.
df_yields = df_yields.reset_index(drop=True)		# index needs to be reset as the columns row is now the header.

In [ ]:
print(df_yields)

### Remove columns

Drop incomplete data from 2025, and last two columns showing summary statistics

In [ ]:
df_yields.columns

First to show rows and columns, can use the iloc index function.

```python
df.iloc[0:10, 0:10] # will show the first 10 rows and first 10 columns.
```
**Reference:** <https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html>

In [ ]:
# shows [rows, columns]
# Top left corner.
df_yields.iloc[:5, :3]

In [ ]:
# showing the last rows of the first three columns.
# Bottom left corner.
df_yields.iloc[10: , :3]

In [ ]:
# shows [rows, columns]
# top right
df_yields.iloc[:5, 23:]

In [ ]:
# shows rows 5 to last, and columns 20 to last.
# Bottom right corner.
df_yields.iloc[10: ,23: ]

#### Trying to remove columns that are not relevant.

- 2025 is not a complete year.
  - Remove this but note future implementation may want to dynamically identify the current year and remove that only. e.g. 2026 will want 2025 data.
- Neitter '% change 2025/2024' nor '2025 confidence interval' are relevant for the analysis.
  - **N.B.** justify why.

In [ ]:
columns_to_drop = [2025, '% change 2025/2024', '2025 confidence interval']
df_yields.drop(columns_to_drop, axis=1, inplace=True)
df_yields.columns

### Rename column for consistency

Rename text column '2022(1)' to numeric 2022.
This was merely a footnote to the data described as follows:

*"(1) Regional areas for 2010 onwards have been replaced with the final regional breakdowns from the June Survey of Agriculture. Regional yields have been recalculated as a result, although any changes are very minor. National areas, national yields and all production figures remain unchanged."*

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
df_yields.rename(columns={"2022(1)":2022}, inplace=True)
df_yields.columns

In [ ]:
print(df_yields)